# TRAIN ViT


Training script for Vision Transformer on a given image dataset (an3/qb3/bnus1/multi) for multilabel classification (BCE loss)
Model checkpoint, train/valid/test split, hyperparameters are given by command line arguments.
The script reports the training curves (loss,accuracy,recall,precision,f1 score) for each of the categories using matplotlib plots and csv outputs
The model is saved using state dict.
"""

In [1]:
#### Only run this once to download the model, BEWARE, it can take a while
# from huggingface_hub import snapshot_download
# snapshot_download(repo_id="google/vit-large-patch16-224", repo_type="model")

## Setup

#### Imports

In [2]:
import torch    #PyTorch
from torch.optim import Adam #The optimizer used (for now)
import torch.nn as nn #submodule that contains the nn.Module class that models inherit from
import numpy as np    #Numpy is used to compute AUPRC using the trapezoidal rule (np.trapz)
import pandas as pd
import math
from torch.utils.data import Dataset 
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms #Transforms such as reshaping and normalizing image pixels w.r to ImageNet.
from transformers import ViTModel, ViTConfig #The library which is used to download and load transformer models
# from transformers import ViTFeatureExtractor, ViTForImageClassification
from torchvision.transforms import ToTensor 
import os
from datetime import datetime
from torchvision.io import read_image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
from torch.utils.tensorboard import SummaryWriter

#### Dataset specific notations

In [3]:
getDefectLabels={
    "AN3": ["splitHinges","missingLeather","visibleNotebook","incompleteCap","incompleteHeadband","strap","box"], 
    "qb3": ["envelope","box","splitHinges","visibleNotebook","incompleteCap"],
    "bnus1": ["splitHinges","missingLeather","visibleNotebook","incompleteCap","incompleteHeadband","strap","box","envelope"],
    "multi": ["splitHinges","missingLeather","visibleNotebook","incompleteCap","incompleteHeadband","strap","box","envelope"]
}

#shorthand notation for the different labels
labels_shortHand={
    "splitHinges":"sH",
    "missingLeather":"mL",
    "visibleNotebook":"vN",
    "incompleteCap":"iC",
    "incompleteHeadband":"iH",
    "strap":"s",
    "box":"b",
    "envelope":"e"
}

### Classes and functions

In [4]:
def mk_timestampedfolder(path, basename, dtime=None):
    if not dtime:
        dtime = datetime.now().strftime('%Y%m%d%H%M')
    
    token = str(hex(int(dtime[2:])))[2:]
    out_folder = os.path.join(path, token + '_' + basename)

    try:
        os.makedirs(out_folder)
    except FileExistsError:
        if not os.listdir(out_folder):
            print('Using existing empty folder')
        else:
            raise
            
    return out_folder

In [5]:
#function to simplify checkpointing
def checkpoint_model(model, filename):
    torch.save(model.state_dict(), filename)

In [6]:
class imgDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None, target_transform=None):
        self.img_labels=pd.read_csv(csv_path)
        self.img_dir=img_dir
        self.transform=transform
        self.target_transform=target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        book_name=self.img_labels.iloc[idx, 0]
        img_path=os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image=read_image(img_path)
        image=image.float()/255
        labels=torch.tensor(self.img_labels.iloc[idx,1:])
        if self.transform:
            image=self.transform(image)
        if self.target_transform:
            labels=self.target_transform(labels)
        return image,labels,book_name


In [7]:
class ViT(nn.Module):

  def __init__(self, num_labels, model_checkpoint, hidden_size, config=ViTConfig()):

        super(ViT, self).__init__()

        self.vit = ViTModel.from_pretrained(model_checkpoint, add_pooling_layer=False, local_files_only=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, num_labels),
            nn.Sigmoid()
        )

  def forward(self, x):

    x = self.vit(x)['last_hidden_state']
    # Use the embedding of [CLS] token
    output = self.classifier(x[:, 0, :])

    return output

#### model_train function – this is where most everything happens, especially tensorboard saves

In [8]:
def model_train(dataset, train_dataloader, valid_dataloader, epochs, learning_rate, bs, checkpoint, hidden_size, model_dir,early_stopping):
    """
    Train a Vision Transformer model for a given number of epochs, learning rate, batch size, and dataset
    (e.g 'an3', 'qb3', 'bnus1' or 'multi')
    Displays the metrics on tensorboard and returns a dataframe with the training/validation metrics once the training is over.
    """

    #creating directory to save model checkpoints
    os.makedirs(os.path.join(model_dir,"checkpoints"),exist_ok=True)
    global_metrics=["loss/train","loss/valid","accuracy/train","accuracy/valid","precision/train","precision/valid","recall/train","recall/valid","F1/train","F1/valid"]
    local_metrics=["validAccuracy","trainingAccuracy","validPrecision","trainingPrecision","validRecall","trainingRecall","validF1","trainingF1"]
    metric_names=global_metrics+[local_metric+"/"+label for local_metric in local_metrics for label in getDefectLabels[dataset]]
    num_labels=len(getDefectLabels[dataset]) #the number of different classification labels in the chosen dataset
    num_metrics=len(metric_names)
    metrics_df=pd.DataFrame(np.zeros((epochs,num_metrics)),columns=metric_names)

    #Tensorboard layout (appears in `custom scalars` menu)
    layout = {
        "Loss and metrics monitoring": {
            "loss": ["Multiline", ["loss/train", "loss/valid"]],
            "overallAccuracy": ["Multiline", ["accuracy/train", "accuracy/valid"]],
            "overallPrecision": ["Multiline", ["precision/train", "precision/valid"]],
            "overallRecall": ["Multiline", ["recall/train", "recall/valid"]],
            "overallF1": ["Multiline", ["F1/train", "F1/valid"]],
        },
    }
    for metric in local_metrics:
        layout["Loss and metrics monitoring"][metric]=["Multiline", [metric+"/"+defect_label for defect_label in getDefectLabels[dataset]]]

    writer.add_custom_scalars(layout)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Load model, loss function, and optimizer
    model = ViT(num_labels=num_labels,model_checkpoint=checkpoint, hidden_size=hidden_size)
    model= nn.DataParallel(model)
    model=model.to(device)
    criterion = nn.BCELoss(reduction='sum').to(device)
    optimizer = Adam(model.parameters(), lr=learning_rate) #training using Adam optimizer
    
    # Fine tuning loop
    best_loss=50 #starting point for best loss
    for epoch_idx in tqdm(range(epochs)):
        model.module.to(device) #sending model to device
        train_accuracy=torch.zeros(num_labels).to(device)
        train_precision=torch.zeros(num_labels).to(device)
        train_recall=torch.zeros(num_labels).to(device)
        train_true_positives=torch.zeros(num_labels,dtype=int).to(device)
        train_gt_positives=torch.zeros(num_labels,dtype=int).to(device)
        train_predicted_positives=torch.zeros(num_labels,dtype=int).to(device)
        train_corrects=torch.zeros(num_labels,dtype=int).to(device)
        train_total_loss=0.

        valid_accuracy=torch.zeros(num_labels).to(device)
        valid_precision=torch.zeros(num_labels).to(device)
        valid_recall=torch.zeros(num_labels).to(device)
        valid_true_positives=torch.zeros(num_labels,dtype=int).to(device)
        valid_gt_positives=torch.zeros(num_labels,dtype=int).to(device)
        valid_predicted_positives=torch.zeros(num_labels,dtype=int).to(device)
        valid_corrects=torch.zeros(num_labels,dtype=int).to(device)
        valid_total_loss=0.
        
        model.train()
        for train_image, train_label, train_book_name in train_dataloader:
            optimizer.zero_grad()
            train_label=train_label.to(device)
            output = model(train_image.to(device))
            train_loss = criterion(output, train_label.float())
            train_loss.backward()
            optimizer.step()

            train_total_loss+=train_loss #computing sum of the losses across batches
            predicted_label=(output>0.5).int() #the predicted labels
            train_corrects+=sum((train_label==predicted_label).int()) #number of correct predictions per class
            train_true_positives+=sum(torch.logical_and((train_label==predicted_label),train_label).int()) #number of true positives per class
            train_gt_positives+=sum(train_label) #number of gt positives per class
            train_predicted_positives+=sum(predicted_label) #number of predicted positives per class

        model.eval()
        with torch.no_grad():
            for valid_image, valid_label, valid_book_name in valid_dataloader:
                valid_label=valid_label.to(device)
                output = model(valid_image.to(device))
                valid_loss = criterion(output, valid_label.float())

                valid_total_loss+=valid_loss #computing sum of the losses across batches
                predicted_label=(output>0.5).int() #the predicted labels
                valid_corrects+=sum((valid_label==predicted_label).int()) #number of correct predictions per category
                valid_true_positives+=sum(torch.logical_and((valid_label==predicted_label),valid_label).int()) #number of true positives epr category
                valid_gt_positives+=sum(valid_label) #number of gt positives per category
                valid_predicted_positives+=sum(predicted_label) #number of predicted positives per category
            
        train_total_loss=train_total_loss/(len(train_dataloader.dataset)) #average loss for the epoch
        train_accuracy=train_corrects/(len(train_dataloader.dataset))
        train_precision=train_true_positives/train_predicted_positives
        train_recall=train_true_positives/train_gt_positives
        train_f1=2*train_precision*train_recall/(train_precision+train_recall) # F1 score

        train_global_accuracy=train_accuracy.mean() #average accuracy among all categories
        train_global_precision=sum(train_true_positives)/sum(train_predicted_positives) #overall training precision
        train_global_recall=sum(train_true_positives)/sum(train_gt_positives) #overall training recall
        train_global_f1=2*train_global_precision*train_global_recall/(train_global_precision+train_global_recall) #overall training f1 score

        valid_total_loss=valid_total_loss/(len(valid_dataloader.dataset)) #average valid loss for the epoch
        valid_accuracy=valid_corrects/(len(valid_dataloader.dataset))
        valid_precision=valid_true_positives/valid_predicted_positives
        valid_recall=valid_true_positives/valid_gt_positives
        valid_f1=2*valid_precision*valid_recall/(valid_precision+valid_recall) # F1 score

        valid_global_accuracy=valid_accuracy.mean() #average accuracy among all labels
        valid_global_precision=sum(valid_true_positives)/sum(valid_predicted_positives) #overall validation precision
        valid_global_recall=sum(valid_true_positives)/sum(valid_gt_positives) #overall validation recall
        valid_global_f1=2*valid_global_precision*valid_global_recall/(valid_global_precision+valid_global_recall) #overall validation f1 score
        
        #Tensorboard logging
        metric_value=train_total_loss.detach().cpu().numpy()
        writer.add_scalar("loss/train",metric_value,epoch_idx)
        metrics_df.at[epoch_idx,"loss/train"]=metric_value
        metric_value=train_global_accuracy.detach().cpu().numpy()
        writer.add_scalar("accuracy/train",metric_value,epoch_idx)
        metrics_df.at[epoch_idx,"accuracy/train"]=metric_value
        metric_value=train_global_precision.detach().cpu().numpy()
        writer.add_scalar("precision/train",metric_value,epoch_idx)
        metrics_df.at[epoch_idx,"precision/train"]=metric_value
        metric_value=train_global_recall.detach().cpu().numpy()
        writer.add_scalar("recall/train",metric_value,epoch_idx)
        metrics_df.at[epoch_idx,"recall/train"]=metric_value
        metric_value=train_global_f1.detach().cpu().numpy()
        writer.add_scalar("F1/train",metric_value,epoch_idx)
        metrics_df.at[epoch_idx,"F1/train"]=metric_value

        metric_value=valid_total_loss.detach().cpu().numpy()
        writer.add_scalar("loss/valid",metric_value,epoch_idx)
        metrics_df.at[epoch_idx,"loss/valid"]=metric_value
        metric_value=valid_global_accuracy.detach().cpu().numpy()
        writer.add_scalar("accuracy/valid",metric_value,epoch_idx)
        metrics_df.at[epoch_idx,"accuracy/valid"]=metric_value
        metric_value=valid_global_precision.detach().cpu().numpy()
        writer.add_scalar("precision/valid",metric_value,epoch_idx)
        metrics_df.at[epoch_idx,"precision/valid"]=metric_value
        metric_value=valid_global_recall.detach().cpu().numpy()
        writer.add_scalar("recall/valid",metric_value,epoch_idx)
        metrics_df.at[epoch_idx,"recall/valid"]=metric_value
        metric_value=valid_global_f1.detach().cpu().numpy()
        writer.add_scalar("F1/valid",metric_value,epoch_idx)
        metrics_df.at[epoch_idx,"F1/valid"]=metric_value

        metric2tensor={
            "validAccuracy": valid_accuracy.detach().cpu().numpy(),
            "trainingAccuracy": train_accuracy.detach().cpu().numpy(),
            "validPrecision": valid_precision.detach().cpu().numpy(),
            "trainingPrecision": train_precision.detach().cpu().numpy(),
            "validRecall": valid_recall.detach().cpu().numpy(),
            "trainingRecall": train_recall.detach().cpu().numpy(),
            "validF1": valid_f1.detach().cpu().numpy(),
            "trainingF1": train_f1.detach().cpu().numpy(),

        }
        for metric in local_metrics:
            for label_idx,label in enumerate(getDefectLabels[dataset]):
                metric_value=metric2tensor[metric][label_idx]
                writer.add_scalar(metric+"/"+label,metric_value,epoch_idx)
                metrics_df.at[epoch_idx,metric+"/"+label]=metric_value
        writer.flush()

        if early_stopping:
            early_stop_loss_threshold=0.2
            if valid_total_loss<best_loss:
                best_loss=valid_total_loss
                checkpoint_model(model.module,os.path.join(model_dir,f"checkpoints/epoch-{epoch_idx}"))
            elif valid_total_loss-best_loss>early_stop_loss_threshold:
                print("Training stopped early because of increase in validation loss.")
                break
        else:
            checkpoint_model(model.module,os.path.join(model_dir,f"checkpoints/epoch-{epoch_idx}"))
    writer.close()
    return model,metrics_df

#### Plotting functions

In [9]:
def plot_train_metrics(metrics_df, save_dir):
    # Beware the many global variables
    epochs=np.arange(len(metrics_df))
    EPOCHS=len(epochs)
    model_name="_".join(save_dir.split("_")[0].split("-")[1:])
    subtitle=model_name+" fined tuned on "+DATASET_NAME+", "+str(int(TRAIN_PERCENT*100))+"/"+str(int(VALID_PERCENT*100))+"/"+str(round((1-VALID_PERCENT-TRAIN_PERCENT)*100))+" train/valid/test split, LR="+str(LR)+", BS="+str(BS)+", EPOCHS="+str(EPOCHS)+", EARLY_STOP="+str(EARLY_STOPPING)
    plots_savepath=os.path.join(save_dir,"plots")
    os.makedirs(plots_savepath,exist_ok=True)

    #plotting loss
    savepath=os.path.join(plots_savepath,"loss.pdf")
    plt.figure()
    plt.scatter(epochs,metrics_df.loc[:,"loss/train"].to_numpy(), marker="o", color='blue', label="Training")
    plt.scatter(epochs,metrics_df.loc[:,"loss/valid"].to_numpy(), marker="s", color='orange', label="Validation")
    plt.legend()
    plt.suptitle("BCE loss per sample (threshold-independant)")
    plt.title(subtitle,fontsize=7)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.savefig(savepath,bbox_inches="tight")
    plt.close()

    #plotting overall accuracy
    savepath=os.path.join(plots_savepath,"overall_accuracy.pdf")
    plt.figure()
    plt.scatter(epochs,metrics_df.loc[:,"accuracy/train"].to_numpy(), marker="o", color='blue', label="Training")
    plt.scatter(epochs,metrics_df.loc[:,"accuracy/valid"].to_numpy(), marker="s", color='orange', label="Validation")
    plt.legend()
    plt.suptitle("Overall accuracy for a τ=50% decision threshold")
    plt.title(subtitle,fontsize=7)
    plt.ylim((-0.05,1.05))
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.savefig(savepath,bbox_inches="tight")
    plt.close()
    
    #plotting overall f1 score
    savepath=os.path.join(plots_savepath,"overall_f1.pdf")
    plt.figure()
    plt.scatter(epochs,metrics_df.loc[:,"F1/train"].to_numpy(), marker="o", color='blue', label="Training")
    plt.scatter(epochs,metrics_df.loc[:,"F1/valid"].to_numpy(), marker="s", color='orange', label="Validation")
    plt.legend()
    plt.suptitle("Overall F1 score for a τ=50% decision threshold")
    plt.title(subtitle,fontsize=7)
    plt.ylim((-0.05,1.05))
    plt.xlabel("Epoch")
    plt.ylabel("F1 score")
    plt.savefig(savepath,bbox_inches="tight")
    plt.close()
    
    #plotting overall precision
    savepath=os.path.join(plots_savepath,"overall_precision.pdf")
    plt.figure()
    plt.scatter(epochs,metrics_df.loc[:,"precision/train"].to_numpy(), marker="o", color='blue', label="Training")
    plt.scatter(epochs,metrics_df.loc[:,"precision/valid"].to_numpy(), marker="s", color='orange', label="Validation")
    plt.legend()
    plt.suptitle("Overall precision for a τ=50% decision threshold")
    plt.title(subtitle,fontsize=7)
    plt.ylim((-0.05,1.05))
    plt.xlabel("Epoch")
    plt.ylabel("Precision")
    plt.savefig(savepath,bbox_inches="tight")
    plt.close()
    
    #plotting overall recall
    savepath=os.path.join(plots_savepath,"overall_recall.pdf")
    plt.figure()
    plt.scatter(epochs,metrics_df.loc[:,"recall/train"].to_numpy(), marker="o", color='blue', label="Training")
    plt.scatter(epochs,metrics_df.loc[:,"recall/valid"].to_numpy(), marker="s", color='orange', label="Validation")
    plt.legend()
    plt.suptitle("Overall recall for a τ=50% decision threshold")
    plt.title(subtitle,fontsize=7)
    plt.ylim((-0.05,1.05))
    plt.xlabel("Epoch")
    plt.ylabel("Recall")
    plt.savefig(savepath,bbox_inches="tight")
    plt.close()
    
    #plotting accuracies
    savepath=os.path.join(plots_savepath,"accuracies.pdf")
    if DATASET_NAME=="qb3":
        fig,ax=plt.subplots(2,3,layout="constrained")
        num_subplots=6
    else:
        fig,ax=plt.subplots(3,3,layout="constrained")
        num_subplots=9
    fig.set_figheight(9.6)
    fig.set_figwidth(12.8)
    plt.setp(ax,xlim=(0,len(metrics_df)-1),ylim=(-0.05,1))
    for k,label in enumerate(getDefectLabels[DATASET_NAME]):
        row_num=k//3
        col_num=k%3
        ax[row_num,col_num].set_title(label)
        ax[row_num,col_num].set_ylim((-0.05,1.05))
        #Plot legend only on last plot
        if k==len(getDefectLabels[DATASET_NAME])-1:
            train_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"trainingAccuracy/"+label].to_numpy(), linewidth=1.5, color='blue')
            valid_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"validAccuracy/"+label].to_numpy(), linewidth=1.5, linestyle="--", color='orange')
            train_plot[0].set_label("Training")
            valid_plot[0].set_label("Validation")
        else:
            train_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"trainingAccuracy/"+label].to_numpy(), linewidth=1.5, color='blue')
            valid_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"validAccuracy/"+label].to_numpy(), linewidth=1.5, linestyle="--", color='orange')

    #not showing unused plots in the grid
    for k in range(len(getDefectLabels[DATASET_NAME]),num_subplots):
        row_num=k//3
        col_num=k%3
        ax[row_num,col_num].remove()

    lines_labels = [ax.get_legend_handles_labels() for ax in fig.axes]
    lines, labels = [sum(lol, []) for lol in zip(*lines_labels)]
    fig.legend(lines,labels,loc="lower right",prop={'size': 15})
    fig.suptitle("Training and validation accuracies for a τ=50% decision threshold \n "+subtitle,weight="bold")
    plt.savefig(savepath,bbox_inches="tight")
    plt.close()

    #plotting F1 scores
    savepath=os.path.join(plots_savepath,"f1s.pdf")
    if DATASET_NAME=="qb3":
        fig,ax=plt.subplots(2,3,layout="constrained")
        num_subplots=6
    else:
        fig,ax=plt.subplots(3,3,layout="constrained")
        num_subplots=9
    fig.set_figheight(9.6)
    fig.set_figwidth(12.8)
    plt.setp(ax,xlim=(0,len(metrics_df)-1),ylim=(-0.05,1))
    for k,label in enumerate(getDefectLabels[DATASET_NAME]):
        row_num=k//3
        col_num=k%3
        ax[row_num,col_num].set_title(label)
        ax[row_num,col_num].set_ylim((-0.05,1.05))
        #Plot legend only on last plot
        if k==len(getDefectLabels[DATASET_NAME])-1:
            train_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"trainingF1/"+label].to_numpy(), linewidth=1.5, color='blue')
            valid_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"validF1/"+label].to_numpy(), linewidth=1.5, linestyle="--", color='orange')
            train_plot[0].set_label("Training")
            valid_plot[0].set_label("Validation")
        else:
            train_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"trainingF1/"+label].to_numpy(), linewidth=1.5, color='blue')
            valid_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"validF1/"+label].to_numpy(), linewidth=1.5, linestyle="--", color='orange')

    #not showing unused plots in the grid
    for k in range(len(getDefectLabels[DATASET_NAME]),num_subplots):
        row_num=k//3
        col_num=k%3
        ax[row_num,col_num].remove()

    lines_labels = [ax.get_legend_handles_labels() for ax in fig.axes]
    lines, labels = [sum(lol, []) for lol in zip(*lines_labels)]
    fig.legend(lines,labels,loc="lower right",prop={'size': 15})
    fig.suptitle("Training and validation F1 scores for a τ=50% decision threshold \n "+subtitle,weight="bold")
    plt.savefig(savepath,bbox_inches="tight")
    plt.close()
    
    #plotting recalls
    savepath=os.path.join(plots_savepath,"recalls.pdf")
    if DATASET_NAME=="qb3":
        fig,ax=plt.subplots(2,3,layout="constrained")
        num_subplots=6
    else:
        fig,ax=plt.subplots(3,3,layout="constrained")
        num_subplots=9
    fig.set_figheight(9.6)
    fig.set_figwidth(12.8)
    plt.setp(ax,xlim=(0,len(metrics_df)-1),ylim=(-0.05,1))
    for k,label in enumerate(getDefectLabels[DATASET_NAME]):
        row_num=k//3
        col_num=k%3
        ax[row_num,col_num].set_title(label)
        ax[row_num,col_num].set_ylim((-0.05,1.05))
        #Plot legend only on last plot
        if k==len(getDefectLabels[DATASET_NAME])-1:
            train_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"trainingRecall/"+label].to_numpy(), linewidth=1, color='blue')
            valid_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"validRecall/"+label].to_numpy(), linewidth=1, linestyle="--", color='orange')
            train_plot[0].set_label("Training")
            valid_plot[0].set_label("Validation")
        else:
            train_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"trainingRecall/"+label].to_numpy(), linewidth=1, color='blue')
            valid_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"validRecall/"+label].to_numpy(), linewidth=1, linestyle="--", color='orange')

    #not showing unused plots in the grid
    for k in range(len(getDefectLabels[DATASET_NAME]),num_subplots):
        row_num=k//3
        col_num=k%3
        ax[row_num,col_num].remove()

    lines_labels = [ax.get_legend_handles_labels() for ax in fig.axes]
    lines, labels = [sum(lol, []) for lol in zip(*lines_labels)]
    fig.legend(lines,labels,loc="lower right",prop={'size': 15})
    fig.suptitle("Training and validation recalls for a τ=50% decision threshold \n "+subtitle,weight="bold")
    plt.savefig(savepath,bbox_inches="tight")
    plt.close()

    #plotting precisions
    savepath=os.path.join(plots_savepath,"precisions.pdf")
    if DATASET_NAME=="qb3":
        fig,ax=plt.subplots(2,3,layout="constrained")
        num_subplots=6
    else:
        fig,ax=plt.subplots(3,3,layout="constrained")
        num_subplots=9
    fig.set_figheight(9.6)
    fig.set_figwidth(12.8)
    plt.setp(ax,xlim=(0,len(metrics_df)-1),ylim=(-0.05,1))
    for k,label in enumerate(getDefectLabels[DATASET_NAME]):
        row_num=k//3
        col_num=k%3
        ax[row_num,col_num].set_title(label)
        ax[row_num,col_num].set_ylim((-0.05,1.05))
        #Plot legend only on last plot
        if k==len(getDefectLabels[DATASET_NAME])-1:
            train_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"trainingPrecision/"+label].to_numpy(), linewidth=1, color='blue')
            valid_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"validPrecision/"+label].to_numpy(), linewidth=1, linestyle="--", color='orange')
            train_plot[0].set_label("Training")
            valid_plot[0].set_label("Validation")
        else:
            train_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"trainingPrecision/"+label].to_numpy(), linewidth=1, color='blue')
            valid_plot=ax[row_num,col_num].plot(epochs,metrics_df.loc[:,"validPrecision/"+label].to_numpy(), linewidth=1, linestyle="--", color='orange')

    #not showing unused plots in the grid
    for k in range(len(getDefectLabels[DATASET_NAME]),num_subplots):
        row_num=k//3
        col_num=k%3
        ax[row_num,col_num].remove()

    lines_labels = [ax.get_legend_handles_labels() for ax in fig.axes]
    lines, labels = [sum(lol, []) for lol in zip(*lines_labels)]
    fig.legend(lines,labels,loc="lower right",prop={'size': 15})
    fig.suptitle("Training and validation precisions for a τ=50% decision threshold \n "+subtitle,weight="bold")
    plt.savefig(savepath,bbox_inches="tight")
    plt.close()

## Training

#### Training Parameters

In [10]:
# DATASET_DIR = The dataset directory being used, e.g /data/zachrodi34/datasets/multi
DATASET_DIR='/Users/csc/Documents/Data/Reliures/AN3'
# CHECKPOINT = The ViT model checkpoint as seen on HuggingFace, e.g 'google/vit-large-patch16-224-in21k' or output of snapshot_download
CHECKPOINT='/Users/csc/.cache/huggingface/hub/models--google--vit-large-patch16-224/snapshots/9e2727f4250d3973839eecfa5c4b42e41b709a50'
# HIDDEN_SIZE = Hidden size in the ViT model (768 for base, 1024 for large, 1280 for huge)
HIDDEN_SIZE=int(1024)
# RESHAPE_SIZE = Reshape size of the input images, depends on the chosen model, often 224
RESHAPE_SIZE=int(224)
# TRAIN_PERCENT = The percentage of the overall dataset to be used for training data, e.g 0.8
TRAIN_PERCENT=float(0.8)
# VALID_PERCENT = The percentage of the overall dataset to be used for validation data, e.g 0.1
VALID_PERCENT=float(0.1)
# EPOCHS = Number of training epochs, e.g 20
EPOCHS=int(20)
# BS = Training Batch size, e.g 64
BS=int(32)
# LR = Learning rate, e.g 1e-5
LR=float(1e-5)
# EARLY_STOPPING = If this flag is applied, early stopping will be used (i.e training is interupted if validation loss increases)
EARLY_STOPPING=True

In [11]:
torch.manual_seed(0) #for reproducibility
writer=SummaryWriter()

In [21]:
DATASET_NAME = os.path.basename(DATASET_DIR)
# MODEL_NAME="_".join([CHECKPOINT.replace("/","-"), DATASET_NAME,"train"+str(TRAIN_PERCENT),"valid"+str(VALID_PERCENT),"epochs"+str(EPOCHS),"bs"+str(BS),"lr"+str(LR),"earlystop"+str(EARLY_STOPPING)])
MODEL_NAME=CHECKPOINT.replace("/","-")
# MODEL_DIR_PATH=os.path.join(DATASET_DIR,MODEL_NAME)
MODEL_OUT_PATH = mk_timestampedfolder(DATASET_DIR, "classification") # create dir that will contain model, csv and plots
MODEL_SAVEPATH=os.path.join(MODEL_OUT_PATH,MODEL_NAME+".pth")
MODEL_CSV_PATH=os.path.join(MODEL_OUT_PATH,MODEL_NAME+"_metrics.csv")

In [13]:
#PREPARING DATA
#Resizing and Normalization transforms need to be applied
MEAN_IMAGENET = [0.485, 0.456, 0.406]
STD_IMAGENET = [0.229, 0.224, 0.225]
transform = transforms.Compose([
    transforms.Resize((RESHAPE_SIZE, RESHAPE_SIZE)),
    transforms.Normalize(mean=MEAN_IMAGENET,std=STD_IMAGENET)
            ])

csv_path=os.path.join(DATASET_DIR,"labels.csv")
img_dir=os.path.join(DATASET_DIR,"books")
full_dataset=imgDataset(csv_path,img_dir,transform)

train_set_size=math.floor(TRAIN_PERCENT*len(full_dataset))
valid_set_size=math.floor(VALID_PERCENT*len(full_dataset))
test_set_size=len(full_dataset)-train_set_size-valid_set_size
train_dataset, valid_test_dataset = torch.utils.data.random_split(full_dataset, [train_set_size, valid_set_size+test_set_size])
valid_dataset, test_dataset = torch.utils.data.random_split(valid_test_dataset, [valid_set_size, test_set_size])
train_dataloader = DataLoader(train_dataset, batch_size=BS, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=BS, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=BS, shuffle=True)

In [16]:
#TRAINING
trained_model, training_dataframe=model_train(DATASET_NAME,train_dataloader,valid_dataloader,EPOCHS,LR,BS,CHECKPOINT, HIDDEN_SIZE, MODEL_OUT_PATH, EARLY_STOPPING)
training_dataframe.to_csv(MODEL_CSV_PATH, index=False) #saving CSV
torch.save(trained_model.module.state_dict(),MODEL_SAVEPATH) #saving model file
plot_train_metrics(training_dataframe,MODEL_OUT_PATH) #create and save plots as pdf files

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: /Users/csc/.cache/huggingface/hub/models--google--vit-large-patch16-224/snapshots/9e2727f4250d3973839eecfa5c4b42e41b709a50
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  0%|                                                    | 0/20 [00:00<?, ?it/s]/var/folders/_f/ns7sh9qd6l549k_h6j8h7x0m0000gn/T/ipykernel_91234/3574931255.py:16: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  labels=torch.tensor(self.img_labels.iloc[idx,1:])
 95%|██████████████████████████████████████  | 19/20 [1:32:13<04:51, 291.22s/it]

Training stopped early because of increasing validation loss.


## Testing

In [17]:
class ViT(nn.Module):

  def __init__(self, config=ViTConfig(), num_labels=7, 
               model_checkpoint=CHECKPOINT):

        super(ViT, self).__init__()

        self.vit = ViTModel.from_pretrained(model_checkpoint, add_pooling_layer=False)
        self.classifier = nn.Sequential(
            nn.Linear(config.hidden_size, num_labels),
            nn.Sigmoid()
        )

  def forward(self, x):

    x = self.vit(x)['last_hidden_state']
    # Use the embedding of [CLS] token
    output = self.classifier(x[:, 0, :])

    return output
      
def get_metrics(dataloader,model,device):
    torch.no_grad()
    model=model.to(device)
    criterion=nn.BCELoss(reduction='sum').to(device)
    accuracy=torch.tensor([0.,0.,0.,0.,0.,0.,0.]).to(device)
    precision=torch.tensor([0.,0.,0.,0.,0.,0.,0.]).to(device)
    recall=torch.tensor([0.,0.,0.,0.,0.,0.,0.]).to(device)
    true_positives=torch.tensor([0,0,0,0,0,0,0]).to(device)
    gt_positives=torch.tensor([0,0,0,0,0,0,0]).to(device)
    predicted_positives=torch.tensor([0,0,0,0,0,0,0]).to(device)
    corrects=torch.tensor([0,0,0,0,0,0,0]).to(device)
    loss=0.
    for image, label in tqdm(dataloader):
            image=image.to(device)
            label=label.to(device)
            output = model(image)
            predicted_label=(output>0.5).int()
            corrects+=sum((label==predicted_label).int())
            true_positives+=sum(torch.logical_and((label==predicted_label),label).int())
            gt_positives+=sum(label)
            predicted_positives+=sum(predicted_label)
            loss+=criterion(output, label.float())

    loss=loss/len(dataloader.dataset)
    accuracy=corrects/(len(dataloader.dataset))
    precision=true_positives/predicted_positives
    recall=true_positives/gt_positives
    f1=2*precision*recall/(precision+recall)
    global_accuracy=torch.mean(accuracy)
    global_precision=sum(true_positives)/sum(predicted_positives)
    global_recall=sum(true_positives)/sum(gt_positives)
    global_f1=2*global_precision*global_recall/(global_precision+global_recall)
    return accuracy,precision,recall,f1,global_accuracy,global_precision,global_recall,global_f1,loss

def get_curves(dataloader,model,device):
    torch.no_grad()
    model=model.to(device)
    criterion=nn.BCELoss(reduction='sum').to(device)
    thresholds=torch.arange(0,1,0.01)
    precision=torch.zeros((100,7)).to(device)
    recall=torch.zeros((100,7)).to(device)
    global_precisions=torch.zeros(100)
    global_recalls=torch.zeros(100)
    global_specificities=torch.zeros(100)
    true_positives=torch.zeros((100,7),dtype=int).to(device)
    gt_positives=torch.zeros((100,7),dtype=int).to(device)
    true_negatives=torch.zeros((100,7),dtype=int).to(device)
    gt_negatives=torch.zeros((100,7),dtype=int).to(device)
    predicted_positives=torch.zeros((100,7),dtype=int).to(device)
    corrects=torch.zeros((100,7),dtype=int).to(device)
    for image, label in tqdm(dataloader):
        image=image.to(device)
        label=label.to(device)
        output = model(image)
        gt_positives[:,:]+=sum(label)
        gt_negatives[:,:]+=sum(torch.logical_not(label))
        for idx,tau in enumerate(thresholds):
            predicted_label=(output>tau).int()
            corrects[idx,:]+=sum((label==predicted_label).int())
            true_positives[idx,:]+=sum(torch.logical_and((label==predicted_label),label).int())
            true_negatives[idx,:]+=sum(torch.logical_and((label==predicted_label),torch.logical_not(label)).int())
            predicted_positives[idx,:]+=sum(predicted_label)

    precision=true_positives/predicted_positives
    recall=true_positives/gt_positives
    specificity=true_negatives/gt_negatives
    
    for idx,tau in enumerate(thresholds):
        global_precision=sum(true_positives[idx,:])/sum(predicted_positives[idx,:])
        global_specificity=sum(true_negatives[idx,:])/sum(gt_negatives[idx,:])
        global_recall=sum(true_positives[idx,:])/sum(gt_positives[idx,:])
        global_precisions[idx]=global_precision
        global_recalls[idx]=global_recall
        global_specificities[idx]=global_specificity

    #correcting problems with precisions when there are no positively predicted for very high threshold values (close to 1 such as 0.996)
    #in that case a division by zero results in nan, which we can replace by 1 for consistency
    precision[precision.isnan()]=1
    global_precisions[global_precisions.isnan()]=1
    
    #recalls are from largest to smallest, which results in negative area
    #we just take the opposite and we obtain the area under the PRC and the ROC

    auprcs=[-np.trapz(precision[:,idx],recall[:,idx]) for idx in range(7)]
    aurocs=[-np.trapz(recall[:,idx],1-specificity[:,idx]) for idx in range(7)]
    global_auprc=-np.trapz(global_precisions,global_recalls)
    global_auroc=-np.trapz(global_recalls,1-global_specificities)
    return precision,recall,specificity,auprcs,aurocs,global_precisions,global_recalls,global_specificities,global_auprc,global_auroc

In [24]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# cpu=torch.device("cpu")
MODEL = "/Users/csc/Documents/Data/Reliures/AN3/9b485d29_classification/-Users-csc-.cache-huggingface-hub-models--google--vit-large-patch16-224-snapshots-9e2727f4250d3973839eecfa5c4b42e41b709a50.pth"
trained_model=ViT()
trained_model.load_state_dict(torch.load(MODEL))
test_accuracy,test_precision,test_recall,test_f1,test_global_accuracy,test_global_precision,test_global_recall,test_global_f1,test_loss=get_metrics(test_dataloader,trained_model,device)
precision,recall,specificity,auprcs,aurocs,global_precisions,global_recalls,global_specificities,global_auprc,global_auroc=get_curves(test_dataloader,trained_model,device)

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: /Users/csc/.cache/huggingface/hub/models--google--vit-large-patch16-224/snapshots/9e2727f4250d3973839eecfa5c4b42e41b709a50
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RuntimeError: Error(s) in loading state_dict for ViT:
	size mismatch for classifier.0.weight: copying a param with shape torch.Size([7, 1024]) from checkpoint, the shape in current model is torch.Size([7, 768]).

### ToDo
- Try SGD optimizer